# Notebook 03: ML Models - Training & Comparison with DLS

Trains 4 ML models (XGBoost, Random Forest, LightGBM, Neural Network) with Optuna tuning,
then compares them against DLS baseline.

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

from src.feature_engineering import (
    load_prepared_data, prepare_features, FEATURE_COLUMNS, TARGET_COLUMN
)
from src.ml_models import (
    train_all_models, predict_with_model, save_model, load_model
)
from src.evaluation import full_evaluation, compute_metrics, generate_latex_table
from src.visualizations import (
    plot_model_comparison_bar, plot_error_distributions,
    plot_phase_rmse, plot_actual_vs_predicted, plot_residuals, plot_learning_curves
)

## 3.1 Load Prepared Data

In [2]:
FORMAT_KEY = 'mens_odi'
train_df, test_df = load_prepared_data(FORMAT_KEY)

print(f"Train set: {len(train_df)} rows, {train_df['match_id'].nunique()} matches")
print(f"Test set: {len(test_df)} rows, {test_df['match_id'].nunique()} matches")
print(f"\nAvailable columns: {list(train_df.columns)}")

Train set: 103930 rows, 2167 matches
Test set: 25392 rows, 542 matches

Available columns: ['match_id', 'batting_team', 'overs_completed', 'overs_remaining', 'current_score', 'wickets_fallen', 'wickets_in_hand', 'current_run_rate', 'recent_run_rate_5', 'scoring_acceleration', 'boundary_percentage', 'dot_ball_percentage', 'partnership_runs', 'is_powerplay', 'is_middle_overs', 'is_death_overs', 'recent_wickets_5', 'cumulative_boundaries', 'cumulative_sixes', 'innings_progress', 'final_total', 'date', 'venue', 'city', 'toss_winner', 'toss_decision', 'year', 'toss_bat_first', 'resource_pct_dls', 'dls_predicted_final']


In [3]:
# Prepare feature matrices
X_train, y_train, feature_names = prepare_features(train_df)
X_test, y_test, _ = prepare_features(test_df)

print(f"Features ({len(feature_names)}): {feature_names}")
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"\nTarget (y_train) stats: mean={y_train.mean():.1f}, std={y_train.std():.1f}")

2026-02-06 17:16:46,193 - INFO - Feature matrix: (103930, 22), Target: (103930,)
2026-02-06 17:16:46,203 - INFO - Feature matrix: (25392, 22), Target: (25392,)


Features (22): ['overs_completed', 'overs_remaining', 'current_score', 'wickets_fallen', 'wickets_in_hand', 'current_run_rate', 'recent_run_rate_5', 'scoring_acceleration', 'boundary_percentage', 'dot_ball_percentage', 'partnership_runs', 'is_powerplay', 'is_middle_overs', 'is_death_overs', 'recent_wickets_5', 'cumulative_boundaries', 'cumulative_sixes', 'innings_progress', 'year', 'toss_bat_first', 'resource_pct_dls', 'dls_predicted_final']
X_train shape: (103930, 22)
X_test shape: (25392, 22)

Target (y_train) stats: mean=248.2, std=64.1


## 3.2 Train All Models with Optuna Tuning

This will take some time due to hyperparameter optimization:
- XGBoost: 100 trials
- Random Forest: 50 trials
- LightGBM: 50 trials
- Neural Network: 30 trials

In [4]:
# Train all models (this may take 30-60 minutes depending on hardware)
model_results = train_all_models(
    X_train, y_train, X_test, y_test,
    format_key=FORMAT_KEY,
    xgb_trials=1,
    rf_trials=1,
    lgb_trials=1,
    nn_trials=0,
)

print("\n✓ All models trained successfully!")
for name, data in model_results.items():
    print(f"  {name}: best trial RMSE = {data['study'].best_value:.2f}")

2026-02-06 17:16:58,650 - INFO - ============================================================
2026-02-06 17:16:58,653 - INFO - Training XGBoost...
2026-02-06 17:16:58,654 - INFO - ============================================================


  0%|          | 0/1 [00:00<?, ?it/s]

2026-02-06 17:17:00,631 - INFO - XGBoost best RMSE: 47.59
2026-02-06 17:17:00,633 - INFO - XGBoost best params: {'n_estimators': 437, 'max_depth': 12, 'learning_rate': 0.1205712628744377, 'subsample': 0.8394633936788146, 'colsample_bytree': 0.6624074561769746, 'min_child_weight': 2, 'reg_alpha': 3.3323645788192616e-08, 'reg_lambda': 0.6245760287469887, 'gamma': 0.0016946556203947059}
2026-02-06 17:17:02,795 - INFO - Saved xgboost to /Users/akshma/Downloads/rajat_thesis/results/models/mens_odi_xgboost.pkl
2026-02-06 17:17:02,800 - INFO - ============================================================
2026-02-06 17:17:02,800 - INFO - Training Random Forest...
2026-02-06 17:17:02,801 - INFO - ============================================================


  0%|          | 0/1 [00:00<?, ?it/s]

2026-02-06 17:17:36,921 - INFO - RF best RMSE: 46.80
2026-02-06 17:17:36,923 - INFO - RF best params: {'n_estimators': 362, 'max_depth': 29, 'min_samples_split': 15, 'min_samples_leaf': 6, 'max_features': 0.40921304830970556}
2026-02-06 17:18:10,615 - INFO - Saved random_forest to /Users/akshma/Downloads/rajat_thesis/results/models/mens_odi_random_forest.pkl
2026-02-06 17:18:10,618 - INFO - ============================================================
2026-02-06 17:18:10,618 - INFO - Training LightGBM...
2026-02-06 17:18:10,619 - INFO - ============================================================


  0%|          | 0/1 [00:00<?, ?it/s]

2026-02-06 17:18:15,926 - INFO - LightGBM best RMSE: 46.51
2026-02-06 17:18:15,927 - INFO - LightGBM best params: {'n_estimators': 437, 'max_depth': 12, 'learning_rate': 0.1205712628744377, 'subsample': 0.8394633936788146, 'colsample_bytree': 0.6624074561769746, 'min_child_samples': 19, 'reg_alpha': 3.3323645788192616e-08, 'reg_lambda': 0.6245760287469887, 'num_leaves': 188}
2026-02-06 17:18:20,760 - INFO - Saved lightgbm to /Users/akshma/Downloads/rajat_thesis/results/models/mens_odi_lightgbm.pkl



✓ All models trained successfully!
  XGBoost: best trial RMSE = 47.59
  RandomForest: best trial RMSE = 46.80
  LightGBM: best trial RMSE = 46.51


## 3.3 Generate Predictions

In [5]:
# Generate predictions for all models
predictions = {}

# DLS baseline
if 'dls_predicted_final' in test_df.columns:
    predictions['DLS'] = test_df['dls_predicted_final'].fillna(test_df['final_total'].mean()).values

# ML models
for name, data in model_results.items():
    model = data['model']
    scaler = data.get('scaler')
    predictions[name] = predict_with_model(model, X_test, scaler=scaler)

# Quick check
for name, preds in predictions.items():
    metrics = compute_metrics(y_test.values, preds)
    print(f"{name}: RMSE={metrics['RMSE']:.2f}, MAE={metrics['MAE']:.2f}, R²={metrics['R2']:.4f}")

DLS: RMSE=61.31, MAE=41.18, R²=0.2443
XGBoost: RMSE=47.59, MAE=34.97, R²=0.5446
RandomForest: RMSE=46.80, MAE=34.04, R²=0.5597
LightGBM: RMSE=46.51, MAE=34.04, R²=0.5651


## 3.4 Full Evaluation & Comparison

In [6]:
# Run full evaluation suite
eval_results = full_evaluation(
    test_df, predictions,
    format_key=FORMAT_KEY,
    overs_limit=50
)

print("\n" + "="*60)
print("OVERALL COMPARISON")
print("="*60)
eval_results['overall']

2026-02-06 17:29:15,834 - INFO - 
Overall Results (mens_odi):
2026-02-06 17:29:15,845 - INFO - 
       Model  Subset  RMSE   MAE     R2  MAPE  Within_5  Within_10  Within_20     N
         DLS Overall 61.31 41.18 0.2443 17.59     13.94      25.86      42.93 25392
     XGBoost Overall 47.59 34.97 0.5446 16.38     12.80      24.39      42.78 25392
RandomForest Overall 46.80 34.04 0.5597 15.79     13.87      25.56      44.32 25392
    LightGBM Overall 46.51 34.04 0.5651 15.92     13.24      24.75      43.72 25392
2026-02-06 17:29:15,879 - INFO - 
Phase-wise Results:
2026-02-06 17:29:15,885 - INFO - 
       Model         Subset   RMSE   MAE      R2  MAPE  Within_5  Within_10  Within_20     N
         DLS   Early (1-10) 101.47 78.12 -0.8760 34.29      4.54       9.45      18.34  5420
     XGBoost   Early (1-10)  68.01 54.26  0.1573 26.60      5.98      12.31      23.93  5420
RandomForest   Early (1-10)  67.67 53.95  0.1657 26.30      6.53      12.80      23.91  5420
    LightGBM   Early (1-


OVERALL COMPARISON


,Model,Subset,RMSE,MAE,R2,MAPE,Within_5,Within_10,Within_20,N
0,DLS,Overall,61.31,41.18,0.2443,17.59,13.94,25.86,42.93,25392
1,XGBoost,Overall,47.59,34.97,0.5446,16.38,12.80,24.39,42.78,25392
2,RandomForest,Overall,46.80,34.04,0.5597,15.79,13.87,25.56,44.32,25392
3,LightGBM,Overall,46.51,34.04,0.5651,15.92,13.24,24.75,43.72,25392


In [7]:
print("PHASE-WISE COMPARISON")
eval_results['phase_wise']

PHASE-WISE COMPARISON


,Model,Subset,RMSE,MAE,R2,MAPE,Within_5,Within_10,Within_20,N
0,DLS,Early (1-10),101.47,78.12,-0.8760,34.29,4.54,9.45,18.34,5420
1,XGBoost,Early (1-10),68.01,54.26,0.1573,26.60,5.98,12.31,23.93,5420
2,RandomForest,Early (1-10),67.67,53.95,0.1657,26.30,6.53,12.80,23.91,5420
3,LightGBM,Early (1-10),66.97,53.40,0.1827,26.22,6.38,12.25,24.30,5420
4,DLS,Middle (11-30),56.56,43.32,0.4011,18.48,9.11,17.57,32.71,10757
5,XGBoost,Middle (11-30),49.76,39.32,0.5366,18.72,8.56,17.28,33.29,10757
6,RandomForest,Middle (11-30),48.74,38.26,0.5552,17.97,8.92,17.57,35.07,10757
7,LightGBM,Middle (11-30),48.48,38.23,0.5600,18.15,8.86,17.83,33.91,10757
8,DLS,Late (31-40),29.56,22.78,0.8033,9.20,14.93,29.95,55.06,4991
9,XGBoost,Late (31-40),30.63,24.07,0.7889,10.25,14.39,27.23,50.61,4991


In [8]:
print("WICKET-STATE COMPARISON")
eval_results['wicket_state']

WICKET-STATE COMPARISON


,Model,Subset,RMSE,MAE,R2,MAPE,Within_5,Within_10,Within_20,N
0,DLS,0-2 wickets,76.37,55.11,-0.1549,22.88,7.66,15.48,29.37,11468
1,XGBoost,0-2 wickets,57.86,44.51,0.3370,19.58,8.31,16.77,32.29,11468
2,RandomForest,0-2 wickets,57.27,43.94,0.3506,19.34,8.39,16.90,32.77,11468
3,LightGBM,0-2 wickets,56.69,43.51,0.3637,19.19,8.48,16.86,32.89,11468
4,DLS,3-5 wickets,52.33,36.31,0.3684,15.69,14.30,25.85,44.55,9148
5,XGBoost,3-5 wickets,41.10,30.76,0.6103,14.68,13.34,25.79,45.78,9148
6,RandomForest,3-5 wickets,40.45,30.04,0.6226,14.28,14.20,26.69,47.33,9148
7,LightGBM,3-5 wickets,40.30,30.14,0.6253,14.41,13.52,25.91,46.40,9148
8,DLS,6-8 wickets,29.46,19.28,0.7928,9.53,25.00,43.84,67.49,3980
9,XGBoost,6-8 wickets,29.20,21.51,0.7965,12.47,20.38,37.46,59.22,3980


## 3.5 Visualizations

In [9]:
# Figure 2: Model comparison bar chart
plot_model_comparison_bar(eval_results['overall'], format_key=FORMAT_KEY)
print("Saved model comparison bar chart")

2026-02-06 17:29:19,124 - INFO - Saved: /Users/akshma/Downloads/rajat_thesis/results/figures/mens_odi_model_comparison_bar.png


Saved model comparison bar chart


In [10]:
# Figure 3: Error distributions
plot_error_distributions(y_test.values, predictions, format_key=FORMAT_KEY)
print("Saved error distributions")

2026-02-06 17:29:20,617 - INFO - Saved: /Users/akshma/Downloads/rajat_thesis/results/figures/mens_odi_error_distributions.png


Saved error distributions


In [11]:
# Figure 4: Phase-wise RMSE
plot_phase_rmse(eval_results['phase_wise'], format_key=FORMAT_KEY)
print("Saved phase RMSE plot")

2026-02-06 17:29:21,005 - INFO - Saved: /Users/akshma/Downloads/rajat_thesis/results/figures/mens_odi_phase_rmse.png


Saved phase RMSE plot


In [12]:
# Figure 8: Actual vs Predicted
plot_actual_vs_predicted(y_test.values, predictions, format_key=FORMAT_KEY)
print("Saved actual vs predicted")

2026-02-06 17:29:23,179 - INFO - Saved: /Users/akshma/Downloads/rajat_thesis/results/figures/mens_odi_actual_vs_predicted.png


Saved actual vs predicted


In [13]:
# Figure 9: Residuals
plot_residuals(y_test.values, predictions, format_key=FORMAT_KEY)
print("Saved residual plots")

2026-02-06 17:29:23,967 - INFO - Saved: /Users/akshma/Downloads/rajat_thesis/results/figures/mens_odi_residuals.png


Saved residual plots


In [14]:
# Figure 10: Learning curves (NN)
if 'NeuralNetwork' in model_results and 'history' in model_results['NeuralNetwork']:
    plot_learning_curves(
        {'NeuralNetwork': model_results['NeuralNetwork']['history']},
        format_key=FORMAT_KEY
    )
    print("Saved learning curves")

## 3.6 LaTeX Tables for Thesis

In [15]:
# Generate LaTeX tables
latex_overall = generate_latex_table(
    eval_results['overall'],
    caption="Overall Model Comparison - Men's ODI",
    label="tab:overall_comparison"
)
print("Overall Comparison LaTeX:")
print(latex_overall)

# Save to file
from src.evaluation import TABLES_DIR
with open(TABLES_DIR / f'{FORMAT_KEY}_overall_latex.tex', 'w') as f:
    f.write(latex_overall)

Overall Comparison LaTeX:
\begin{table}[htbp]
\centering
\caption{Overall Model Comparison - Men's ODI}
\label{tab:overall_comparison}
\begin{tabular}{lrrrrrrrrr}
\toprule
Model & Subset & RMSE & MAE & R2 & MAPE & Within_5 & Within_10 & Within_20 & N \\
\midrule
DLS & Overall & 61.31 & 41.18 & 0.24 & 17.59 & 13.94 & 25.86 & 42.93 & 25392 \\
XGBoost & Overall & 47.59 & 34.97 & 0.54 & 16.38 & 12.80 & 24.39 & 42.78 & 25392 \\
RandomForest & Overall & 46.80 & 34.04 & 0.56 & 15.79 & 13.87 & 25.56 & 44.32 & 25392 \\
LightGBM & Overall & 46.51 & 34.04 & 0.57 & 15.92 & 13.24 & 24.75 & 43.72 & 25392 \\
\bottomrule
\end{tabular}
\end{table}


In [16]:
# Improvement over DLS
overall = eval_results['overall']
dls_rmse = overall[overall['Model'] == 'DLS']['RMSE'].values

if len(dls_rmse) > 0:
    dls_rmse = dls_rmse[0]
    print(f"DLS Baseline RMSE: {dls_rmse:.2f}")
    print("\nImprovement over DLS:")
    for _, row in overall.iterrows():
        if row['Model'] != 'DLS':
            improvement = ((dls_rmse - row['RMSE']) / dls_rmse) * 100
            print(f"  {row['Model']}: {improvement:+.1f}% RMSE reduction")

print("\n✓ ML model training and comparison complete!")

DLS Baseline RMSE: 61.31

Improvement over DLS:
  XGBoost: +22.4% RMSE reduction
  RandomForest: +23.7% RMSE reduction
  LightGBM: +24.1% RMSE reduction

✓ ML model training and comparison complete!
